# NRPy in Ten Minutes

Follow one small pipeline from a symbolic expression to a registered C function.

Navigation: [Index](../index.ipynb) | Previous: [Getting Started with NRPy](../0-getting_started/install_and_run.ipynb) | Next: [Parameters](params.ipynb)


## Learning Goals

- See the full NRPy workflow in miniature.
- Connect a symbolic energy expression to generated C code.
- Recognize that NRPy remembers named fields and functions while building a project.

## Words for This Notebook

- **Symbolic expression:** a formula represented by Python objects instead of immediate decimal numbers.
- **Grid field:** a quantity sampled at many grid points, like a wave value at each point.
- **Registry:** NRPy's in-memory list of named objects created during the notebook.
- **C function:** a named block of C code that can be called later.

Use the code cells actively: first predict what should happen, then run the cell, then explain the output in plain language. This predict-run-explain pattern keeps the physics idea connected to the programming details.


## One Miniature Pipeline
The expression below is a symbolic wave-energy density. NRPy turns it into C, then stores a full function in NRPy's function list.


## Import SymPy for the Toy Expression

These imports expose the NRPy and Python tools used in the next steps.


In [1]:
import sympy as sp


## Import NRPy Core Registries

These imports expose the NRPy and Python tools used in the next steps.


In [2]:
import nrpy.c_codegen as ccg
from nrpy.c_function import CFunction_dict, register_CFunction
import nrpy.grid as grid
import nrpy.params as par


## Clear Tutorial Registry Entries

This reset removes only the names introduced by this overview so rerunning the notebook starts from a known state.


In [3]:
CFunction_dict.pop("overview_energy_density", None)
for name in ["uu", "vv"]:
    grid.glb_gridfcs_dict.pop(name, None)
par.set_parval_from_str("Infrastructure", "BHaH")


## Register Wave Variables and Parameters

NRPy's stored grid-field list records named fields and their roles in generated code.


In [4]:
uu, vv = grid.register_gridfunctions(["uu", "vv"], group="EVOL")
wave_speed = par.register_CodeParameter(
    "REAL", "tutorial_overview", "overview_wave_speed", 1.0
)
energy_density = sp.Rational(1, 2) * (vv**2 + wave_speed**2 * uu**2)
assignment = ccg.c_codegen(
    energy_density, "energy_density", include_braces=False, verbose=False
)
print(assignment)


energy_density = (1.0/2.0)*((overview_wave_speed)*(overview_wave_speed))*((uu)*(uu)) + (1.0/2.0)*((vv)*(vv));



## Register and Inspect a Tiny C Function

NRPy's stored function list holds a complete C function or workflow hook for later file generation.


In [5]:
register_CFunction(
    name="overview_energy_density",
    desc="Compute a compact wave-energy diagnostic.",
    cfunc_type="void",
    params="const double uu, const double vv, const double wave_speed, double *energy",
    body="*energy = 0.5*(vv*vv + wave_speed*wave_speed*uu*uu);\nreturn;",
)
print("registered keys:", sorted(CFunction_dict))
print(CFunction_dict["overview_energy_density"].function_prototype)


registered keys: ['overview_energy_density']
void overview_energy_density(const double uu, const double vv, const double wave_speed, double *energy);


The output compresses the main workflow: register fields, build a symbolic expression, emit C, then register a complete C function. The named stored function can later be written into source files.


## Learning Check

Before running the registration cells, predict which names should appear in NRPy's memory: the wave fields, the energy expression, or both? After running, check your prediction.


## Continue to Parameters
- [Parameters](params.ipynb)
- [Indexed Expressions](indexedexp.ipynb)
- [Gridfunctions and Grid Metadata](grid.ipynb)
- [C Code Generation](c_codegen.ipynb)
- [C Function Registration](c_function.ipynb)
- [Wave Equation and C Code Generation](../3-wave_equation/wave_equation_and_c_codegen.ipynb)
